<a href="https://colab.research.google.com/github/yasumorishima/mlb-data-analysis/blob/main/notebooks/sql/ohtani_injury_analysis_2023_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shohei Ohtani Injury Precursor Analysis 2023 (SQL Version)

Statistical analysis of pitching metrics to detect potential injury warning signs using **DuckDB SQL**.

(大谷翔平の2023年シーズンにおける怪我予兆検出の**SQL版**統計分析)

## SQL Skills Demonstrated:
- `AVG`, `STDDEV` aggregate functions
- Window functions for time-series analysis
- `CASE WHEN` for anomaly detection (±2σ)
- Date filtering and grouping
- Subqueries for baseline comparison

In [ ]:
!pip install pybaseball duckdb -q

In [ ]:
from pybaseball import statcast
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## 1. Data Acquisition (データ取得)

Ohtani's 2023 pitching data (pitcher ID: 660271)

In [ ]:
# Fetch 2023 season Statcast data
df = statcast(start_dt='2023-03-30', end_dt='2023-08-31')
print(f"Total records: {len(df):,}")

In [ ]:
# DuckDB connection
con = duckdb.connect()

## 2. Extract Ohtani's Pitching Data (大谷の投球データ抽出)

In [ ]:
# Get Ohtani's pitching appearances
df_ohtani = con.execute("""
    SELECT *
    FROM df
    WHERE pitcher = 660271
      AND pitch_type IS NOT NULL
    ORDER BY game_date, at_bat_number, pitch_number
""").df()

print(f"Ohtani's 2023 pitches: {len(df_ohtani):,}")

In [ ]:
# Get unique game dates (start dates)
game_dates = con.execute("""
    SELECT DISTINCT game_date
    FROM df
    WHERE pitcher = 660271
    ORDER BY game_date
""").df()

print(f"Number of starts: {len(game_dates)}")
print(game_dates)

## 3. Calculate Baseline Statistics (ベースライン統計量の算出)

Using data before June 27, 2023 as baseline.

In [ ]:
# Calculate baseline statistics (before June 27)
baseline_stats = con.execute("""
    SELECT 
        pitch_type,
        ROUND(AVG(release_speed), 2) as avg_speed,
        ROUND(STDDEV(release_speed), 2) as std_speed,
        ROUND(AVG(release_spin_rate), 0) as avg_spin,
        ROUND(STDDEV(release_spin_rate), 0) as std_spin,
        ROUND(AVG(release_pos_x), 3) as avg_release_x,
        ROUND(STDDEV(release_pos_x), 3) as std_release_x,
        ROUND(AVG(release_pos_z), 3) as avg_release_z,
        ROUND(STDDEV(release_pos_z), 3) as std_release_z,
        COUNT(*) as pitch_count
    FROM df
    WHERE pitcher = 660271
      AND game_date < '2023-06-27'
      AND pitch_type IS NOT NULL
    GROUP BY pitch_type
    ORDER BY pitch_count DESC
""").df()

print("Baseline Statistics (before June 27):")
print(baseline_stats.to_string(index=False))

## 4. Daily Metrics Tracking (日別指標の追跡)

In [ ]:
# Calculate daily averages for key metrics
daily_metrics = con.execute("""
    SELECT 
        game_date,
        ROUND(AVG(release_speed), 2) as avg_speed,
        ROUND(AVG(release_spin_rate), 0) as avg_spin,
        ROUND(AVG(release_pos_x), 3) as avg_release_x,
        ROUND(AVG(release_pos_z), 3) as avg_release_z,
        ROUND(AVG(release_extension), 3) as avg_extension,
        COUNT(*) as pitch_count
    FROM df
    WHERE pitcher = 660271
      AND pitch_type IS NOT NULL
    GROUP BY game_date
    ORDER BY game_date
""").df()

print("Daily Metrics:")
print(daily_metrics)

## 5. Anomaly Detection (異常検知 - ±2σ Method)

In [ ]:
# Detect anomalies using ±2σ method with SQL
anomaly_detection = con.execute("""
    WITH baseline AS (
        SELECT 
            AVG(release_speed) as baseline_speed,
            STDDEV(release_speed) as std_speed,
            AVG(release_spin_rate) as baseline_spin,
            STDDEV(release_spin_rate) as std_spin,
            AVG(release_pos_x) as baseline_x,
            STDDEV(release_pos_x) as std_x,
            AVG(release_pos_z) as baseline_z,
            STDDEV(release_pos_z) as std_z
        FROM df
        WHERE pitcher = 660271
          AND game_date < '2023-06-27'
          AND pitch_type IS NOT NULL
    ),
    daily_avg AS (
        SELECT 
            game_date,
            AVG(release_speed) as daily_speed,
            AVG(release_spin_rate) as daily_spin,
            AVG(release_pos_x) as daily_x,
            AVG(release_pos_z) as daily_z
        FROM df
        WHERE pitcher = 660271
          AND pitch_type IS NOT NULL
        GROUP BY game_date
    )
    SELECT 
        d.game_date,
        ROUND(d.daily_speed, 2) as speed,
        CASE 
            WHEN d.daily_speed < b.baseline_speed - 2 * b.std_speed THEN 'LOW'
            WHEN d.daily_speed > b.baseline_speed + 2 * b.std_speed THEN 'HIGH'
            ELSE 'NORMAL'
        END as speed_status,
        ROUND(d.daily_spin, 0) as spin,
        CASE 
            WHEN d.daily_spin < b.baseline_spin - 2 * b.std_spin THEN 'LOW'
            WHEN d.daily_spin > b.baseline_spin + 2 * b.std_spin THEN 'HIGH'
            ELSE 'NORMAL'
        END as spin_status,
        ROUND(d.daily_x, 3) as release_x,
        CASE 
            WHEN d.daily_x < b.baseline_x - 2 * b.std_x THEN 'LOW'
            WHEN d.daily_x > b.baseline_x + 2 * b.std_x THEN 'HIGH'
            ELSE 'NORMAL'
        END as x_status,
        ROUND(d.daily_z, 3) as release_z,
        CASE 
            WHEN d.daily_z < b.baseline_z - 2 * b.std_z THEN 'LOW'
            WHEN d.daily_z > b.baseline_z + 2 * b.std_z THEN 'HIGH'
            ELSE 'NORMAL'
        END as z_status
    FROM daily_avg d
    CROSS JOIN baseline b
    ORDER BY d.game_date
""").df()

print("Anomaly Detection Results:")
print(anomaly_detection.to_string(index=False))

## 6. Count Anomalies Over Time (異常発生数の推移)

In [ ]:
# Count anomalies per game
anomaly_counts = con.execute("""
    WITH baseline AS (
        SELECT 
            AVG(release_speed) as baseline_speed,
            STDDEV(release_speed) as std_speed,
            AVG(release_spin_rate) as baseline_spin,
            STDDEV(release_spin_rate) as std_spin,
            AVG(release_pos_x) as baseline_x,
            STDDEV(release_pos_x) as std_x,
            AVG(release_pos_z) as baseline_z,
            STDDEV(release_pos_z) as std_z
        FROM df
        WHERE pitcher = 660271
          AND game_date < '2023-06-27'
          AND pitch_type IS NOT NULL
    ),
    pitch_anomalies AS (
        SELECT 
            d.game_date,
            CASE WHEN ABS(d.release_speed - b.baseline_speed) > 2 * b.std_speed THEN 1 ELSE 0 END as speed_anomaly,
            CASE WHEN ABS(d.release_spin_rate - b.baseline_spin) > 2 * b.std_spin THEN 1 ELSE 0 END as spin_anomaly,
            CASE WHEN ABS(d.release_pos_x - b.baseline_x) > 2 * b.std_x THEN 1 ELSE 0 END as x_anomaly,
            CASE WHEN ABS(d.release_pos_z - b.baseline_z) > 2 * b.std_z THEN 1 ELSE 0 END as z_anomaly
        FROM df d
        CROSS JOIN baseline b
        WHERE d.pitcher = 660271
          AND d.pitch_type IS NOT NULL
    )
    SELECT 
        game_date,
        COUNT(*) as total_pitches,
        SUM(speed_anomaly) as speed_anomalies,
        SUM(spin_anomaly) as spin_anomalies,
        SUM(x_anomaly) as x_anomalies,
        SUM(z_anomaly) as z_anomalies,
        SUM(speed_anomaly + spin_anomaly + x_anomaly + z_anomaly) as total_anomalies,
        ROUND(SUM(speed_anomaly + spin_anomaly + x_anomaly + z_anomaly) * 100.0 / COUNT(*), 1) as anomaly_rate
    FROM pitch_anomalies
    GROUP BY game_date
    ORDER BY game_date
""").df()

print("Anomaly Counts per Game:")
print(anomaly_counts.to_string(index=False))

## 7. Visualization (可視化)

In [ ]:
# Get pitch-level data with anomaly flags using SQL
df_plot = con.execute("""
    WITH baseline AS (
        SELECT
            AVG(release_pos_x) as baseline_x,
            STDDEV(release_pos_x) as std_x,
            AVG(release_spin_rate) as baseline_spin,
            STDDEV(release_spin_rate) as std_spin
        FROM df
        WHERE pitcher = 660271
          AND game_date < '2023-06-27'
          AND pitch_type IS NOT NULL
          AND release_pos_x IS NOT NULL
          AND release_spin_rate IS NOT NULL
    )
    SELECT
        d.game_date,
        d.release_pos_x,
        d.release_spin_rate,
        b.baseline_x,
        b.std_x,
        b.baseline_spin,
        b.std_spin,
        CASE
            WHEN ABS(d.release_pos_x - b.baseline_x) > 2 * b.std_x
              OR ABS(d.release_spin_rate - b.baseline_spin) > 2 * b.std_spin
            THEN 1 ELSE 0
        END as is_anomaly,
        CASE WHEN d.game_date < '2023-06-27' THEN 'Before June 27' ELSE 'After June 27' END as period
    FROM df d
    CROSS JOIN baseline b
    WHERE d.pitcher = 660271
      AND d.pitch_type IS NOT NULL
      AND d.release_pos_x IS NOT NULL
      AND d.release_spin_rate IS NOT NULL
    ORDER BY d.game_date
""").df()

# Plot: Release Position X vs Spin Rate (±2σ anomaly detection)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, period in enumerate(['Before June 27', 'After June 27']):
    period_data = df_plot[df_plot['period'] == period]
    normal = period_data[period_data['is_anomaly'] == 0]
    anomaly = period_data[period_data['is_anomaly'] == 1]

    # Plot normal points
    axes[i].scatter(normal['release_pos_x'], normal['release_spin_rate'],
                    alpha=0.5, c='blue', label='Normal', s=20)
    # Plot anomaly points (outside ±2σ)
    axes[i].scatter(anomaly['release_pos_x'], anomaly['release_spin_rate'],
                    alpha=0.8, c='red', label='Anomaly (±2σ)', s=30)

    # Draw ±2σ boundary box
    if len(period_data) > 0:
        bx, sx = period_data['baseline_x'].iloc[0], period_data['std_x'].iloc[0]
        bs, ss = period_data['baseline_spin'].iloc[0], period_data['std_spin'].iloc[0]
        rect = plt.Rectangle((bx - 2*sx, bs - 2*ss), 4*sx, 4*ss,
                              fill=False, edgecolor='green', linestyle='--', linewidth=2, label='±2σ boundary')
        axes[i].add_patch(rect)

    anomaly_pct = len(anomaly) / len(period_data) * 100 if len(period_data) > 0 else 0
    axes[i].set_title(f'{period}\n(Anomaly Rate: {anomaly_pct:.1f}%)')
    axes[i].set_xlabel('Release Position X (ft)')
    axes[i].set_ylabel('Spin Rate (rpm)')
    axes[i].legend(loc='upper right')

plt.suptitle('Shohei Ohtani 2023 - Release Position × Spin Rate Anomaly Detection', fontsize=14)
plt.tight_layout()
plt.show()

# Print anomaly rate comparison
print("\n=== Anomaly Rate Comparison ===")
for period in ['Before June 27', 'After June 27']:
    period_data = df_plot[df_plot['period'] == period]
    anomaly_count = period_data['is_anomaly'].sum()
    total = len(period_data)
    print(f"{period}: {anomaly_count}/{total} pitches ({anomaly_count/total*100:.1f}%) outside ±2σ")

## Key Findings

1. **Increasing Anomalies**: The anomaly rate tends to increase as the season progresses
2. **Multi-parameter Analysis**: Combining release position × spin rate may reveal early warning signs
3. **±2σ Method**: Statistical outlier detection helps identify potential injury precursors

## SQL Techniques Used

- **CTEs**: Organized baseline calculation and anomaly detection
- **CASE WHEN**: Classified each metric as NORMAL/HIGH/LOW
- **Aggregate Functions**: AVG, STDDEV, COUNT, SUM
- **CROSS JOIN**: Applied baseline to all rows for comparison
- **Date Filtering**: Separated pre/post baseline periods